In [ ]:
import sys, os
import numpy as np
import pandas as pd
import pauc
import scipy
from scipy import stats
import choix
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import seaborn as sns

from csc import *
from exp_utils import *

current_dir = os.path.dirname(os.path.abspath('__file__'))
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [2]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scienceplots
plt.style.use(['science', 'no-latex'])
plt.rcParams['text.latex.preamble'] = r'\usepackage[cm]{sfmath}'
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'

Run `collect_entropy.py` before this notebook.

In [3]:
potato_duplicate_questions = [14, 83, 121]
models = [
    "gemma-2-9b-it",
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Phi-3.5-mini-instruct",
]

model_rename = {
    "gemma-2-9b-it": "Gemma-2-9B",
    "gemma-3-12b-it": "Gemma-3-12B",
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B",
    "Mistral-7B-Instruct-v0.3": "Mistral-v0.3-7B",
    "Phi-3.5-mini-instruct": "Phi-3.5-3.8B",
}

uncertainty_methods = {
    "hybrid-alphabet": "$\\widehat{|S|}_{Hybrid}$",
    "NumSets": "NumSets",
    "gt": "Good-Turing",
    "ueigv": "$U_{EigV}$",
    "cs-hybrid": "$\\widehat{\\mathbb{H}}_{Hybrid}$",
    "plugin": "$\\widehat{\\mathbb{H}}_{Plugin}$",
    "cs": "$\\widehat{\\mathbb{H}}_{CS}$",
    "predictive": "PE",
    "snne": "SNNE",
    "kle": "KLE",
    "se": "SE",
}

datasets = {
    "hotpot_qa_final": "HotpotQA",
    "squad_v2_final": "SQuAD 2.0 (Answerable)",
    # "potato_final": "POTATO",
    "bioasq_final": "BioASQ",
}
pm_symbol = u"\u00B1"

p = "preprompt"

In [4]:
fname = f"{current_dir}/data/{p}/{models[1]}/{"squad_v2_final"}_results.json"
with open(fname) as f:
    summary = json.load(f)

In [5]:
squad_all_ids = [int(i) for i in summary]
squad_answerable_ids = [int(i) for i in summary if summary[i]["answerable"]]
squad_unanswerable_ids = [int(i) for i in summary if not summary[i]["answerable"]]

In [ ]:
import tol_colors as tc
colors = tc.get_colorset('muted')
methods_palette = {
    uncertainty_methods["hybrid-alphabet"]: colors.wine,
    uncertainty_methods["NumSets"]: colors.cyan,
    uncertainty_methods["gt"]: colors.indigo,
    uncertainty_methods["ueigv"]: colors.olive,
    uncertainty_methods["cs-hybrid"]: colors.rose,
    uncertainty_methods["plugin"]: "grey", 
    uncertainty_methods["cs"]: colors.sand, 
    uncertainty_methods["predictive"]: colors.pale_grey,
    uncertainty_methods["snne"]: colors.teal,
    uncertainty_methods["kle"]: colors.purple,
    uncertainty_methods["se"]: "mediumseagreen",
}

#### Incorrectness AUC

In [7]:
def get_incorrectness_df(judge_threshold=70, n=10, squad_answerable_only=True):
    barplot_data = {
        "Dataset": [],
        "Model": [],
        "Uncertainty Measure": [],
        "AUROC": [],
        "Interval Lower": [],
        "Interval Upper": [],
        "Err": [],
    }
    stars = []
    test = []
    for model in models:
        uncertainty_df = pd.read_csv(f"{current_dir}/data/{p}/{model}/uncertainty.csv")
        uncertainty_df["incorrect"] = (uncertainty_df["judge-llm-score"]<=judge_threshold).astype(int) # predicts INcorrectness
        for method in uncertainty_methods:
            for dataset in datasets:
                uncertainty_df_subset = uncertainty_df[(uncertainty_df["dataset"]==dataset)&(uncertainty_df["n"]==n)&(uncertainty_df["judge-llm-score"]!=-1)]
                
                if squad_answerable_only and dataset == "squad_v2_final":
                    uncertainty_df_subset = uncertainty_df_subset[uncertainty_df_subset["id"].isin(squad_answerable_ids)]
                
                method_scores = uncertainty_df_subset[method].values

                ground_truth = uncertainty_df_subset["incorrect"].values
                
                auc, (lb, ub) = pauc.roc_auc_ci.roc_auc_ci_score(
                    ground_truth,
                    method_scores,
                )

                barplot_data["Dataset"].append(datasets[dataset])
                barplot_data["Model"].append(model_rename[model])
                barplot_data["Uncertainty Measure"].append(uncertainty_methods[method])
                barplot_data["AUROC"].append(auc)
                barplot_data["Interval Lower"].append(lb)
                barplot_data["Interval Upper"].append(ub)
                barplot_data["Err"].append(ub-auc)
                test.append(auc-lb==ub-auc)

    return pd.DataFrame(barplot_data)

In [ ]:
fig = plt.figure(figsize=(13, 4))
gs = gridspec.GridSpec(1, 3, figure=fig)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])
axes = [ax1, ax2, ax3]
axes_labels = ["A", "B", "C"]

large_fontsize = 20
medium_fontsize = 17.5
small_fontsize = 15

df = get_incorrectness_df(judge_threshold=70, n=10, squad_answerable_only=True)

idx = 0
for dataset in datasets:
    ax = axes[idx]
    subset = df[df["Dataset"]==datasets[dataset]]

    ax = sns.barplot(
        subset, 
        x="Model", 
        y="AUROC", 
        hue="Uncertainty Measure",
        ax=ax,
        edgecolor="black",
        linewidth=0.05,
        palette=methods_palette.values(),
        legend=False,
    )

    x_coords = [p.get_x() + 0.5*p.get_width() for p in ax.patches]
    y_coords = [p.get_height() for p in ax.patches]
    ax.errorbar(
        x=x_coords, 
        y=y_coords, 
        yerr=subset["Err"], 
        fmt="none", 
        color="black", 
        capsize=0
    )
    
    ax.set_title(datasets[dataset], fontsize=large_fontsize)

    ax.set_ylim(bottom=0.4)

    ax.set(xlabel=None)
    if idx == 0:
        ax.set_ylabel("AUROC", fontsize=large_fontsize)
    else:
        ax.set_ylabel(None)
    ax.tick_params(axis='both', which='major', labelsize=small_fontsize)

    for a in ax.get_xticklabels():
        a.set_rotation(25)
        a.set_fontsize(medium_fontsize)
        a.set_horizontalalignment("right")
        a.set_rotation_mode("anchor")

    sns.despine(top=True, right=True, left=False, bottom=False, ax=ax)
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.yaxis.set_minor_locator(plt.NullLocator())
    ax.tick_params(axis="y", which="major", right=False)
    ax.tick_params(axis="x", which="major", top=False) 

    idx += 1

legend_items = {k: Patch(facecolor=v, label=k) for k,v in methods_palette.items()}
fig.legend(
    title=None,
    labels=legend_items.keys(),
    handles=legend_items.values(),
    bbox_to_anchor=(0.5, -0.35),
    fancybox=False, 
    shadow=False, 
    ncol=len(uncertainty_methods)/2+1,
    loc='lower center',
    fontsize=medium_fontsize
)

plt.tight_layout()
plt.savefig('figures/incorrectness_preprompt_draft.pdf')

#### Win Rate

In [ ]:
df = get_incorrectness_df(judge_threshold=70, n=10, squad_answerable_only=True)

In [10]:
def gaussian_samples(lb, ub, mean, n=100):
    margin = (ub-lb)/2
    sigma = margin/1.96
    samples = np.random.normal(loc=mean, scale=sigma, size=n)
    return samples

In [11]:
# win rate
direct_winrate_matrix = np.ones((len(uncertainty_methods), len(uncertainty_methods)))*100
methods_list = list(uncertainty_methods.values())

for i in range(len(methods_list)):
    method1 = methods_list[i]
    for j in range(len(methods_list)):
        method2 = methods_list[j]
        if method1 == method2:
            continue

        # % of time method 1 has higher AUROC than method 2
        comparisons = (df[df["Uncertainty Measure"]==method1]["AUROC"].values>df[df["Uncertainty Measure"]==method2]["AUROC"].values)
        win_rate = 100*comparisons.sum()/len(comparisons)
        
        direct_winrate_matrix[i, j] = win_rate

In [12]:
# Bradley-Terry (Point Estimates)

matches = []
for i in range(len(methods_list)):
    method1 = methods_list[i]
    method1_df = df[df["Uncertainty Measure"]==method1]

    for j in range(i, len(methods_list)):
        method2 = methods_list[j]
        if method1 == method2:
            continue
        method2_df = df[df["Uncertainty Measure"]==method2]

        for model in model_rename.values():
            for dataset in datasets.values():
                method1_auroc = method1_df[(method1_df["Model"]==model)&(method1_df["Dataset"]==dataset)]["AUROC"].item()
                method2_auroc = method2_df[(method2_df["Model"]==model)&(method2_df["Dataset"]==dataset)]["AUROC"].item()

                if method1_auroc > method2_auroc:
                    matches.append((i, j))
                if method1_auroc < method2_auroc:
                    matches.append((j, i))

tournament_point = Tournament(matches=matches, entities=methods_list, L=1)
scores_point = tournament_point.get_mle_scores(alpha=0.1, format="array")

bt_results_point = {}
for i in range(len(methods_list)):
    bt_results_point[methods_list[i]] = scores_point[i]

bt_results_df_point = pd.DataFrame(bt_results_point.items(), columns=["Uncertainty Measure", "Strength"])


# Bradley-Terry (Uncertainty-Aware)

matches = []
n_trials = 1000
for i in range(len(methods_list)):
    method1 = methods_list[i]
    method1_df = df[df["Uncertainty Measure"]==method1]

    for j in range(i, len(methods_list)):
        method2 = methods_list[j]
        if method1 == method2:
            continue
        method2_df = df[df["Uncertainty Measure"]==method2]

        for model in model_rename.values():
            for dataset in datasets.values():
                method1_samples = gaussian_samples(
                    lb=method1_df[(method1_df["Model"]==model)&(method1_df["Dataset"]==dataset)]["Interval Lower"].item(),
                    ub=method1_df[(method1_df["Model"]==model)&(method1_df["Dataset"]==dataset)]["Interval Upper"].item(),
                    mean=method1_df[(method1_df["Model"]==model)&(method1_df["Dataset"]==dataset)]["AUROC"].item(),
                    n=n_trials
                )
                method2_samples = gaussian_samples(
                    lb=method2_df[(method2_df["Model"]==model)&(method2_df["Dataset"]==dataset)]["Interval Lower"].item(),
                    ub=method2_df[(method2_df["Model"]==model)&(method2_df["Dataset"]==dataset)]["Interval Upper"].item(),
                    mean=method2_df[(method2_df["Model"]==model)&(method2_df["Dataset"]==dataset)]["AUROC"].item(),
                    n=n_trials
                )

                for k in range(n_trials):
                    method1_auroc = method1_samples[k]
                    method2_auroc = method2_samples[k]

                    if method1_auroc > method2_auroc:
                        matches.append((i, j))
                    if method1_auroc < method2_auroc:
                        matches.append((j, i))

tournament_mc = Tournament(matches=matches, entities=methods_list, L=100)
scores_mc = tournament_mc.get_mle_scores(alpha=0.1, format="array")
errs_mc = tournament_mc.get_mle_errs(alpha=0.1, pct=0.95, conservative=True)

bt_results_mc = {}
for i in range(len(methods_list)):
    bt_results_mc[methods_list[i]] = scores_mc[i]

bt_results_df_mc = pd.DataFrame(bt_results_mc.items(), columns=["Uncertainty Measure", "Strength"])

In [ ]:
fig = plt.figure(figsize=(6, 6))
gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[20, 1])
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0])
axes_labels = ["A"]

# Win Rate
mask = mask_diagonal = np.eye(direct_winrate_matrix.shape[0], dtype=bool)
ax1 = sns.heatmap(
    direct_winrate_matrix,
    mask=mask,
    # linewidth=0.5, 
    cmap="RdBu", 
    annot=True, 
    fmt=".0f",
    ax=ax1,
    annot_kws={"size": small_fontsize-2.5},
    cbar_kws={'orientation': 'horizontal'},
    cbar_ax=ax2
)

ax1.set_xticklabels(methods_list, ha='right')
for a in ax1.get_xticklabels():
    a.set_rotation(45)
    a.set_fontsize(small_fontsize)
    a.set_horizontalalignment("right")
    a.set_rotation_mode("anchor")

ax1.set_yticklabels(methods_list, rotation=0, ha='right')
for a in ax1.get_yticklabels():
    a.set_fontsize(small_fontsize)

sns.despine(top=True, right=True, left=True, bottom=True, ax=ax1)
ax1.xaxis.set_minor_locator(plt.NullLocator())
ax1.yaxis.set_minor_locator(plt.NullLocator())
ax1.tick_params(axis="y", which="major", right=False)
ax1.tick_params(axis="x", which="major", top=False)

cbar = ax1.collections[0].colorbar
cbar.minorticks_off()
cbar.ax.tick_params(labelsize=small_fontsize-2.5)

plt.tight_layout()
plt.savefig('figures/tournament_winrate_draft.pdf')

In [ ]:
fig = plt.figure(figsize=(8, 4))
gs = gridspec.GridSpec(1, 2, figure=fig)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
axes = [ax1, ax2]
axes_labels = ["A", "B"]

# Bradley-Terry (Point Estimates)

ax1 = sns.barplot(
    bt_results_df_point, 
    x="Uncertainty Measure", 
    y="Strength",
    hue="Uncertainty Measure",
    ax=ax1,
    edgecolor="black",
    linewidth=0.75,
    palette=methods_palette.values(),
    legend=False
)

ax1.set_ylabel("Latent strength", fontsize=large_fontsize)
ax1.set_xlabel(None)

ax1.set_xticklabels(methods_list, ha='right')
for a in ax1.get_xticklabels():
    a.set_rotation(45)
    a.set_fontsize(small_fontsize)
    a.set_horizontalalignment("right")
    a.set_rotation_mode("anchor")

sns.despine(top=True, right=True, left=False, bottom=False, ax=ax1)
ax1.xaxis.set_minor_locator(plt.NullLocator())
ax1.yaxis.set_minor_locator(plt.NullLocator())
ax1.tick_params(axis="y", which="major", right=False)
ax1.tick_params(axis="x", which="major", top=False) 

ax1.text(0, 1.1, 
    f'A (Point Estimates)',
    transform=ax1.transAxes, 
    verticalalignment='top', 
    horizontalalignment='left', 
    fontweight='bold',
    fontsize=large_fontsize,
)


# Bradley-Terry (Monte Carlo)

ax2 = sns.barplot(
    bt_results_df_mc, 
    x="Uncertainty Measure", 
    y="Strength",
    hue="Uncertainty Measure",
    ax=ax2,
    edgecolor="black",
    linewidth=0.75,
    palette=methods_palette.values(),
    legend=False
)

x_coords = [p.get_x() + 0.5*p.get_width() for p in ax2.patches]
y_coords = [p.get_height() for p in ax2.patches]
ax2.errorbar(
    x=x_coords, 
    y=y_coords, 
    yerr=errs_mc, 
    fmt="none", 
    color="black", 
    capsize=0
)

ax2.set_ylabel(None)
ax2.set_xlabel(None)

ax2.set_xticklabels(methods_list, ha='right')
for a in ax2.get_xticklabels():
    a.set_rotation(45)
    a.set_fontsize(small_fontsize)
    a.set_horizontalalignment("right")
    a.set_rotation_mode("anchor")

sns.despine(top=True, right=True, left=False, bottom=False, ax=ax2)
ax2.xaxis.set_minor_locator(plt.NullLocator())
ax2.yaxis.set_minor_locator(plt.NullLocator())
ax2.tick_params(axis="y", which="major", right=False)
ax2.tick_params(axis="x", which="major", top=False) 

ax2.text(0, 1.1, 
    f'B (Monte Carlo)',
    transform=ax2.transAxes, 
    verticalalignment='top', 
    horizontalalignment='left', 
    fontweight='bold',
    fontsize=large_fontsize,
)

plt.tight_layout()
plt.savefig('figures/tournament_bt_draft.pdf')

#### Bradley-Terry Rank

In [16]:
alpha_vals = [0, 0.01, 0.1, 1]

tournament_mc_sensitivity = {}

for alpha in alpha_vals:
    for i in range(tournament_mc.n):
        entity = tournament_mc.entities[i]
        if entity not in tournament_mc_sensitivity:
            tournament_mc_sensitivity[entity] = []
        
        interval = tournament_mc.get_rank_ci(i_target=i, alpha=alpha, pct=0.95)
        tournament_mc_sensitivity[entity].append(interval)

In [17]:
tournament_mc_sensitivity_df = pd.DataFrame.from_dict(
    data=tournament_mc_sensitivity, 
    orient="index", 
    columns=[f"${a}$" for a in alpha_vals]
)

In [19]:
latex = tournament_mc_sensitivity_df[["$0.1$"]].to_latex(
    index=True,
)
print(latex)

\begin{tabular}{ll}
\toprule
 & $0.1$ \\
\midrule
$\widehat{|S|}_{Hybrid}$ & [1, 4] \\
NumSets & [7, 11] \\
Good-Turing & [7, 11] \\
$U_{EigV}$ & [1, 3] \\
$\widehat{\mathbb{H}}_{Hybrid}$ & [5, 5] \\
$\widehat{\mathbb{H}}_{Plugin}$ & [7, 11] \\
$\widehat{\mathbb{H}}_{CS}$ & [7, 11] \\
PE & [3, 4] \\
SNNE & [6, 6] \\
KLE & [1, 3] \\
SE & [7, 11] \\
\bottomrule
\end{tabular}



In [20]:
latex = tournament_mc_sensitivity_df.to_latex(
    index=True,
)
print(latex)

\begin{tabular}{lllll}
\toprule
 & $0$ & $0.01$ & $0.1$ & $1$ \\
\midrule
$\widehat{|S|}_{Hybrid}$ & [1, 4] & [1, 4] & [1, 4] & [1, 4] \\
NumSets & [7, 11] & [7, 11] & [7, 11] & [7, 11] \\
Good-Turing & [7, 11] & [7, 11] & [7, 11] & [7, 11] \\
$U_{EigV}$ & [1, 3] & [1, 3] & [1, 3] & [1, 3] \\
$\widehat{\mathbb{H}}_{Hybrid}$ & [5, 5] & [5, 5] & [5, 5] & [5, 5] \\
$\widehat{\mathbb{H}}_{Plugin}$ & [7, 11] & [7, 11] & [7, 11] & [7, 11] \\
$\widehat{\mathbb{H}}_{CS}$ & [7, 11] & [7, 11] & [7, 11] & [7, 11] \\
PE & [3, 4] & [3, 4] & [3, 4] & [3, 4] \\
SNNE & [6, 6] & [6, 6] & [6, 6] & [6, 6] \\
KLE & [1, 3] & [1, 3] & [1, 3] & [1, 3] \\
SE & [7, 11] & [7, 11] & [7, 11] & [7, 11] \\
\bottomrule
\end{tabular}

